# 💬 InvestAI — Análisis de Sentimiento NLP con VADER

**Proyecto:** Sistema Ernesto Investing AI  
**Módulo:** Análisis de Sentimiento de Noticias Financieras  
**Fuente:** Yahoo Finance News API (vía `yfinance`) + `vaderSentiment`  
**Salida:** MongoDB Atlas (`sentimiento_noticias`, `sentimiento_resumen`)  

---

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) es un
analizador de sentimiento basado en léxico, diseñado específicamente
para textos de redes sociales y noticias. Devuelve cuatro scores:

| Score | Descripción |
|---|---|
| `pos` | Proporción de tokens positivos |
| `neg` | Proporción de tokens negativos |
| `neu` | Proporción de tokens neutros |
| `compound` | Score global normalizado en [-1.0, +1.0] |

**Umbral de decisión:**  
- `compound ≥ +0.05` → Sentimiento **POSITIVO** (señal alcista)  
- `compound ≤ -0.05` → Sentimiento **NEGATIVO** (señal bajista)  
- Entre ambos → **NEUTRO**

> ⚠️ **Requisito:** `MONGO_URI` configurado en los Secrets de Colab.

In [ ]:
import requests

# Obtener la IP pública actual de la sesión de Colab
response = requests.get('https://api.ipify.org?format=json')
public_ip = response.json()['ip']
print(f"Tu dirección IP pública actual de Colab es: {public_ip}")

Tu dirección IP pública actual de Colab es: 34.81.114.169


## Sección 1 — Instalación de Dependencias

In [ ]:
# vaderSentiment: analizador de sentimiento léxico para textos financieros
# yfinance: descarga de noticias por ticker desde Yahoo Finance
# pymongo[srv]: cliente MongoDB con soporte Atlas
!pip install vaderSentiment yfinance "pymongo[srv]" --quiet

# Descargar lexicón VADER si es necesario
import nltk
nltk.download('vader_lexicon', quiet=True)

print("✅ Dependencias instaladas correctamente.")

## Sección 2 — Importación de Librerías

In [ ]:
import warnings, math, time
warnings.filterwarnings('ignore')
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yfinance as yf

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from pymongo import MongoClient, UpdateOne
from pymongo.errors import ConnectionFailure
from google.colab import userdata

# Inicializar el analizador VADER
analizador = SentimentIntensityAnalyzer()

print("✅ Librerías importadas y analizador VADER inicializado.")

## Sección 3 — Configuración Global

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

EMPRESAS_META = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',              'moneda': 'USD'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compañía Minera S.A.A.',          'moneda': 'PEN'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',               'moneda': 'CAD'},
    'BVN':         {'nombre': 'Compañía de Minas Buenaventura S.A.A.',  'moneda': 'USD'},
    'BHP':         {'nombre': 'BHP Group Limited',                      'moneda': 'USD'},
}

# ─── Umbrales VADER estándar para textos financieros ──────────────────────
UMBRAL_POSITIVO =  0.05   # compound >= +0.05 → POSITIVO
UMBRAL_NEGATIVO = -0.05   # compound <= -0.05 → NEGATIVO

# Pausa entre peticiones a Yahoo Finance (respeto de rate limits)
PAUSA_SEGUNDOS = 1.5

# ─── MongoDB ──────────────────────────────────────────────────────────────
DB_NOMBRE          = 'investai'
COL_NOTICIAS       = 'sentimiento_noticias'   # Una entrada por noticia
COL_RESUMEN        = 'sentimiento_resumen'    # Score agregado por ticker

print("✅ Configuración cargada.")
print(f"   Umbral positivo : compound ≥ {UMBRAL_POSITIVO}")
print(f"   Umbral negativo : compound ≤ {UMBRAL_NEGATIVO}")

## Sección 4 — Conexión a MongoDB Atlas

In [ ]:
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("MONGO_URI vacío.")
except Exception as e:
    raise RuntimeError(f"Secret MONGO_URI no encontrado: {e}")

try:
    cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10_000)
    cliente.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida.")
except ConnectionFailure as e:
    raise RuntimeError(f"No se pudo conectar: {e}")

db           = cliente[DB_NOMBRE]
col_noticias = db[COL_NOTICIAS]
col_resumen  = db[COL_RESUMEN]

# Índice compuesto único para noticias: (ticker, uuid o titulo+fecha)
# Evita duplicados si se ejecuta el notebook más de una vez
col_noticias.create_index(
    [('ticker', 1), ('uuid', 1)],
    unique=True, name='idx_ticker_uuid',
    sparse=True   # sparse=True permite documentos sin 'uuid'
)
# Índice único para el resumen: un documento por ticker
col_resumen.create_index([('ticker', 1)], unique=True, name='idx_ticker')

print(f"   Colección noticias : {COL_NOTICIAS}")
print(f"   Colección resumen  : {COL_RESUMEN}")

## Sección 5 — Funciones de Análisis

In [ ]:
def nan_a_none(v):
    """Convierte NaN/Inf a None para serialización JSON-segura en MongoDB."""
    if v is None:
        return None
    try:
        return None if (math.isnan(v) or math.isinf(v)) else round(float(v), 6)
    except (TypeError, ValueError):
        return None


def clasificar_sentimiento(compound: float) -> str:
    """
    Clasifica el score compound de VADER en tres categorías
    usando los umbrales estándar para texto financiero.

    Parámetros:
        compound : score VADER en [-1.0, +1.0]

    Retorna:
        'POSITIVO', 'NEGATIVO' o 'NEUTRO'
    """
    if compound >= UMBRAL_POSITIVO:
        return 'POSITIVO'
    elif compound <= UMBRAL_NEGATIVO:
        return 'NEGATIVO'
    return 'NEUTRO'


def analizar_texto(texto: str) -> dict:
    """
    Aplica VADER sobre un texto y devuelve los scores con clasificación.

    Parámetros:
        texto : titular o resumen de noticia

    Retorna:
        dict con pos, neg, neu, compound y etiqueta de sentimiento
    """
    if not texto or not texto.strip():
        return {'pos': None, 'neg': None, 'neu': None,
                'compound': None, 'sentimiento': 'NEUTRO'}

    scores = analizador.polarity_scores(texto)
    return {
        'pos'        : nan_a_none(scores['pos']),
        'neg'        : nan_a_none(scores['neg']),
        'neu'        : nan_a_none(scores['neu']),
        'compound'   : nan_a_none(scores['compound']),
        'sentimiento': clasificar_sentimiento(scores['compound']),
    }


def obtener_noticias_yfinance(ticker: str) -> list[dict]:
    """
    Descarga noticias recientes de Yahoo Finance para un ticker.

    yfinance.Ticker.news devuelve una lista de dicts con campos:
        uuid, title, publisher, link, providerPublishTime, type, thumbnail

    Parámetros:
        ticker : símbolo bursátil

    Retorna:
        lista de dicts de noticias, o lista vacía si hay error
    """
    try:
        activo  = yf.Ticker(ticker)
        noticias = activo.news
        return noticias if noticias else []
    except Exception as e:
        print(f"   ⚠️  {ticker}: error descargando noticias → {e}")
        return []


print("✅ Funciones de análisis definidas:")
print("   · clasificar_sentimiento(compound)")
print("   · analizar_texto(texto)")
print("   · obtener_noticias_yfinance(ticker)")

## Sección 6 — Funciones de Persistencia

In [ ]:
def guardar_noticias(ticker: str, documentos: list[dict]) -> dict:
    """
    Guarda las noticias analizadas en 'sentimiento_noticias' con upsert.
    Usa (ticker, uuid) como clave única para idempotencia.

    Parámetros:
        ticker     : símbolo bursátil
        documentos : lista de docs generados por el pipeline

    Retorna:
        dict con estadísticas de la operación
    """
    if not documentos:
        return {'insertadas': 0, 'actualizadas': 0}

    operaciones = [
        UpdateOne(
            filter={'ticker': doc['ticker'], 'uuid': doc.get('uuid', doc['titulo'])},
            update={'$set': doc},
            upsert=True
        )
        for doc in documentos
    ]
    res = col_noticias.bulk_write(operaciones, ordered=False)
    return {'insertadas': res.upserted_count, 'actualizadas': res.modified_count}


def guardar_resumen(ticker: str, resumen: dict) -> None:
    """
    Upsert del resumen de sentimiento por ticker en 'sentimiento_resumen'.
    Un único documento por ticker, siempre actualizado.

    El resumen contiene el score promedio, la distribución de
    sentimientos y la señal de sentimiento vigente.

    Parámetros:
        ticker  : símbolo bursátil
        resumen : dict con métricas agregadas
    """
    meta = EMPRESAS_META.get(ticker, {})
    doc = {
        'ticker'            : ticker,
        'nombre'            : meta.get('nombre', ticker),
        'n_noticias'        : resumen['n_noticias'],
        'compound_promedio' : resumen['compound_promedio'],
        'compound_mediana'  : resumen['compound_mediana'],
        'sentimiento_global': resumen['sentimiento_global'],
        'distribucion'      : resumen['distribucion'],
        'noticias_recientes': resumen['noticias_recientes'],
        'actualizado_en'    : datetime.now(timezone.utc),
    }
    col_resumen.update_one(
        filter={'ticker': ticker},
        update={'$set': doc},
        upsert=True
    )


print("✅ Funciones de persistencia definidas.")

## Sección 7 — Pipeline Principal: Análisis de Sentimiento por Ticker

In [ ]:
resumen_global = []
timestamp_ingesta = datetime.now(timezone.utc)

print("=" * 70)
print("  ANÁLISIS DE SENTIMIENTO VADER — InvestAI")
print("=" * 70)

for ticker in TICKERS:
    meta = EMPRESAS_META.get(ticker, {})
    print(f"\n💬 {ticker}  ({meta.get('nombre', '')})")

    # ── Paso 1: Descarga de noticias desde Yahoo Finance ──────────────────
    noticias_raw = obtener_noticias_yfinance(ticker)
    time.sleep(PAUSA_SEGUNDOS)   # Respetar rate limits de Yahoo Finance

    if not noticias_raw:
        print(f"   ⚠️  Sin noticias disponibles para {ticker}.")
        resumen_global.append({
            'ticker': ticker, 'n_noticias': 0,
            'sentimiento': '—', 'compound': None
        })
        continue

    print(f"   Noticias encontradas : {len(noticias_raw)}")

    # ── Paso 2: Análisis VADER de cada noticia ────────────────────────────
    documentos = []
    scores_compound = []

    for noticia in noticias_raw:
        # Extraer campos de la noticia (estructura de yfinance.news)
        titulo    = noticia.get('title', '')
        uuid      = noticia.get('uuid', '')
        publisher = noticia.get('publisher', '')
        link      = noticia.get('link', '')
        ts_pub    = noticia.get('providerPublishTime')

        # Convertir timestamp Unix a datetime UTC
        fecha_pub = (
            datetime.fromtimestamp(ts_pub, tz=timezone.utc)
            if ts_pub else None
        )

        # Analizar el titular con VADER
        # (el summary completo no siempre está disponible en la API gratuita)
        vader_titulo = analizar_texto(titulo)

        if vader_titulo['compound'] is not None:
            scores_compound.append(vader_titulo['compound'])

        doc = {
            'ticker'         : ticker,
            'nombre'         : meta.get('nombre', ticker),
            'uuid'           : uuid or titulo[:80],   # Fallback si no hay uuid
            'titulo'         : titulo,
            'publisher'      : publisher,
            'link'           : link,
            'fecha_publicacion': fecha_pub,
            'fecha_pub_str'  : fecha_pub.strftime('%Y-%m-%d') if fecha_pub else None,
            # Scores VADER del titular
            'vader_pos'      : vader_titulo['pos'],
            'vader_neg'      : vader_titulo['neg'],
            'vader_neu'      : vader_titulo['neu'],
            'vader_compound' : vader_titulo['compound'],
            'sentimiento'    : vader_titulo['sentimiento'],
            'analizado_en'   : timestamp_ingesta,
        }
        documentos.append(doc)

    # ── Paso 3: Cálculo del resumen agregado por ticker ────────────────────
    if scores_compound:
        compound_prom  = float(np.mean(scores_compound))
        compound_med   = float(np.median(scores_compound))
        sent_global    = clasificar_sentimiento(compound_prom)
    else:
        compound_prom  = None
        compound_med   = None
        sent_global    = 'NEUTRO'

    # Distribución de sentimientos
    n_pos  = sum(1 for d in documentos if d['sentimiento'] == 'POSITIVO')
    n_neg  = sum(1 for d in documentos if d['sentimiento'] == 'NEGATIVO')
    n_neu  = sum(1 for d in documentos if d['sentimiento'] == 'NEUTRO')
    n_tot  = len(documentos)

    distribucion = {
        'positivo': n_pos,
        'negativo': n_neg,
        'neutro'  : n_neu,
        'pct_pos' : nan_a_none(n_pos / n_tot * 100) if n_tot else None,
        'pct_neg' : nan_a_none(n_neg / n_tot * 100) if n_tot else None,
    }

    # Las 3 noticias más recientes con sus scores (para mostrar en el frontend)
    noticias_recientes = [
        {
            'titulo'    : d['titulo'],
            'publisher' : d['publisher'],
            'fecha'     : d['fecha_pub_str'],
            'compound'  : d['vader_compound'],
            'sentimiento': d['sentimiento'],
            'link'      : d['link'],
        }
        for d in sorted(
            documentos,
            key=lambda x: x['fecha_publicacion'] or datetime.min.replace(tzinfo=timezone.utc),
            reverse=True
        )[:5]
    ]

    resumen_ticker = {
        'n_noticias'        : n_tot,
        'compound_promedio' : nan_a_none(compound_prom),
        'compound_mediana'  : nan_a_none(compound_med),
        'sentimiento_global': sent_global,
        'distribucion'      : distribucion,
        'noticias_recientes': noticias_recientes,
    }

    print(f"   Noticias analizadas  : {n_tot}")
    print(f"   Compound promedio    : {compound_prom}")
    print(f"   Sentimiento global   : {sent_global}")
    print(f"   Distribución         : +{n_pos} / -{n_neg} / ~{n_neu}")

    # ── Paso 4: Guardado en MongoDB ────────────────────────────────────────
    stats = guardar_noticias(ticker, documentos)
    guardar_resumen(ticker, resumen_ticker)

    print(f"   ✅ MongoDB → noticias: +{stats['insertadas']} nuevas, "
          f"{stats['actualizadas']} actualizadas | resumen: upsert OK")

    resumen_global.append({
        'ticker'     : ticker,
        'n_noticias' : n_tot,
        'compound'   : compound_prom,
        'sentimiento': sent_global,
        'pos / neg / neu': f"{n_pos} / {n_neg} / {n_neu}",
    })

print("\n" + "=" * 70)
print("  RESUMEN FINAL")
print("=" * 70)
print(pd.DataFrame(resumen_global).to_string(index=False))
print("=" * 70)

## Sección 8 — Verificación Final

In [ ]:
print("=" * 70)
print("  VERIFICACIÓN — Consulta a MongoDB Atlas")
print("=" * 70)

# ── 1. Noticias individuales ───────────────────────────────────────────────
n_noticias = col_noticias.count_documents({})
print(f"\n📰 sentimiento_noticias : {n_noticias:,} documentos totales")

for ticker in TICKERS:
    n = col_noticias.count_documents({'ticker': ticker})
    print(f"   {ticker:<14} : {n} noticias")

# ── 2. Resúmenes por ticker ────────────────────────────────────────────────
print(f"\n📊 sentimiento_resumen : {col_resumen.count_documents({})} documentos")
for doc in col_resumen.find({}, {'_id': 0, 'noticias_recientes': 0}).sort('ticker', 1):
    print(f"   {doc['ticker']:<14}  "
          f"sent: {doc['sentimiento_global']:<10}  "
          f"compound: {doc.get('compound_promedio', '?')}  "
          f"noticias: {doc.get('n_noticias', '?')}")

# ── 3. Ejemplo de noticia ─────────────────────────────────────────────────
print("\n📄 Ejemplo de noticia analizada (BVN, más reciente):")
ej = col_noticias.find_one(
    {'ticker': 'BVN'},
    sort=[('fecha_publicacion', -1)],
    projection={'_id': 0, 'link': 0}
)
if ej:
    for k, v in ej.items():
        if isinstance(v, datetime):
            v = v.strftime('%Y-%m-%d %H:%M UTC')
        print(f"   {k:<22}: {v}")
else:
    print("   ⚠️  No se encontró ejemplo para BVN.")

print("\n✅ Verificación completada.")

In [ ]:
cliente.close()
print("✅ Conexión a MongoDB Atlas cerrada.")